# Clean Wind Validation (CF vs CF)

This notebook restarts the validation using like-for-like metrics:
- compare **capacity factor to capacity factor** (not raw MWh)
- run both hourly and daily comparisons
- report best lag in ±24 h to detect alignment issues

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from scipy import stats

MODEL_ROOT = Path().resolve().parent
if str(MODEL_ROOT) not in sys.path:
    sys.path.insert(0, str(MODEL_ROOT))

from src.models.generation import power_curve_parametric, extrapolate_wind_speed
from src.data.loaders import (
    load_alpha_ventus_wind,
    load_frauenhofer,
    load_pt_generation,
    load_capacity_all,
)

YEAR = 2023
print('Imports OK')

In [ ]:
def lag_scan(model_series, actual_series, max_lag=24):
    out = []
    for lag in range(-max_lag, max_lag + 1):
        r = model_series.shift(lag).corr(actual_series)
        out.append((lag, r, r**2 if pd.notna(r) else np.nan))
    lag_df = pd.DataFrame(out, columns=['lag_h', 'r', 'r2'])
    best = lag_df.loc[lag_df['r'].abs().idxmax()]
    return lag_df, int(best['lag_h']), float(best['r']), float(best['r2'])

def quick_metrics(tag, model, actual):
    slope, intercept, r, p, _ = stats.linregress(model, actual)
    print(tag)
    print(f'  r={r:.4f}  R²={r**2:.4f}  slope={slope:.3f}  intercept={intercept:.3f}')
    return slope, intercept, r, r**2

## Test A (Portugal): WFA model CF vs measured WFA CF

In [ ]:
# Open-Meteo wind at WFA location (UTC)
url = (
    'https://archive-api.open-meteo.com/v1/archive'
    '?latitude=41.85&longitude=-9.03'
    f'&start_date={YEAR}-01-01&end_date={YEAR}-12-31'
    '&hourly=wind_speed_100m&wind_speed_unit=ms'
)
resp = requests.get(url, timeout=30)
resp.raise_for_status()
j = resp.json()

wfa = pd.DataFrame({
    'timestamp': pd.to_datetime(j['hourly']['time']).tz_localize('UTC').tz_localize(None),
    'wind_speed_ms': j['hourly']['wind_speed_100m'],
})

# Turbine model -> model CF
v_hub = extrapolate_wind_speed(wfa['wind_speed_ms'].to_numpy(), z_ref=100.0, z_hub=110.0, z0=0.0002)
wfa['cf_model'] = power_curve_parametric(v_hub, cut_in_ms=3.5, rated_speed_ms=14.0, cut_out_ms=25.0)

# Portugal actual offshore generation (MTU marked CET/CEST) -> UTC
pt = load_pt_generation(YEAR).copy()
pt_ts = pd.to_datetime(pt['timestamp'], errors='coerce')
pt_ts = pt_ts.dt.tz_localize('Europe/Berlin', ambiguous='NaT', nonexistent='shift_forward')
pt['timestamp'] = pt_ts.dt.tz_convert('UTC').dt.tz_localize(None)
pt = pt.dropna(subset=['timestamp']).reset_index(drop=True)

# Measured CF: WFA is the only offshore farm in PT (25.2 MW)
PT_WFA_CAP_MW = 25.2
pt['cf_actual'] = pt['generation_mwh'] / PT_WFA_CAP_MW
pt['cf_actual'] = pt['cf_actual'].clip(lower=0, upper=1.2)

a = pd.merge(wfa[['timestamp', 'cf_model']], pt[['timestamp', 'cf_actual']], on='timestamp', how='inner').dropna()

print(f'Merged rows: {len(a):,}')
print(f'Mean CF model={a.cf_model.mean():.3f}, actual={a.cf_actual.mean():.3f}')

In [ ]:
# Hourly + daily metrics and lag scan
quick_metrics('Test A hourly (CF vs CF)', a['cf_model'], a['cf_actual'])
lag_a, best_lag_a, best_r_a, best_r2_a = lag_scan(a['cf_model'], a['cf_actual'])
print(f'  best lag={best_lag_a:+d}h  best R²={best_r2_a:.4f}')

a_day = a.set_index('timestamp').resample('D').mean(numeric_only=True).dropna().reset_index()
quick_metrics('Test A daily (CF vs CF)', a_day['cf_model'], a_day['cf_actual'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(a['cf_model'], a['cf_actual'], s=6, alpha=0.15, color='steelblue')
axes[0].set_title('Test A hourly CF scatter')
axes[0].set_xlabel('Model CF')
axes[0].set_ylabel('Actual CF')
axes[0].grid(True, alpha=0.3)

axes[1].plot(a_day['timestamp'], a_day['cf_model'], label='Model CF', color='steelblue')
axes[1].plot(a_day['timestamp'], a_day['cf_actual'], label='Actual CF', color='tomato')
axes[1].set_title('Test A daily CF')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('CF')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## Test B (Germany): Alpha Ventus model CF vs German fleet measured CF

In [ ]:
av = load_alpha_ventus_wind(YEAR).copy()
av['timestamp'] = pd.to_datetime(av['timestamp']).dt.tz_localize(None)
v_hub_av = extrapolate_wind_speed(av['wind_speed_ms'].to_numpy(), z_ref=100.0, z_hub=92.0, z0=0.0002)
av['cf_model'] = power_curve_parametric(v_hub_av, cut_in_ms=3.5, rated_speed_ms=14.0, cut_out_ms=25.0)

de = load_frauenhofer(YEAR).copy()
de['timestamp'] = pd.to_datetime(de['timestamp'])

cap = load_capacity_all().copy()
cap = cap[(cap['year_num'] == YEAR) & cap['month_num'].between(1, 12)][['month_num', 'Wind offshore']].rename(columns={'Wind offshore':'cap_gw'})

de['month_num'] = de['timestamp'].dt.month
de = de.merge(cap, on='month_num', how='left')
de['cf_actual'] = de['generation_mwh'] / (de['cap_gw'] * 1000.0)
de['cf_actual'] = de['cf_actual'].clip(lower=0, upper=1.2)

b = pd.merge(av[['timestamp', 'cf_model']], de[['timestamp', 'cf_actual']], on='timestamp', how='inner').dropna()

print(f'Merged rows: {len(b):,}')
print(f'Mean CF model={b.cf_model.mean():.3f}, actual={b.cf_actual.mean():.3f}')

In [ ]:
quick_metrics('Test B hourly (CF vs CF)', b['cf_model'], b['cf_actual'])
lag_b, best_lag_b, best_r_b, best_r2_b = lag_scan(b['cf_model'], b['cf_actual'])
print(f'  best lag={best_lag_b:+d}h  best R²={best_r2_b:.4f}')

b_day = b.set_index('timestamp').resample('D').mean(numeric_only=True).dropna().reset_index()
quick_metrics('Test B daily (CF vs CF)', b_day['cf_model'], b_day['cf_actual'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(b['cf_model'], b['cf_actual'], s=5, alpha=0.12, color='darkorange')
axes[0].set_title('Test B hourly CF scatter')
axes[0].set_xlabel('Model CF')
axes[0].set_ylabel('Actual CF')
axes[0].grid(True, alpha=0.3)

axes[1].plot(b_day['timestamp'], b_day['cf_model'], label='Model CF', color='darkorange')
axes[1].plot(b_day['timestamp'], b_day['cf_actual'], label='Actual CF', color='steelblue')
axes[1].set_title('Test B daily CF')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('CF')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## Power-output mapping check (MWh vs MWh)\n\nThis section keeps the original idea: map wind to power output and compare against measured output.\n\nTo avoid hidden mistakes, it adds:\n- saturation check (how often model is at/near rated output)\n- bias check (mean model minus mean actual)\n- chronological train/test scaling (fit on first 70%, validate on last 30%)\n\nIf train/test both improve with one simple scale factor, the shape is good and mismatch is mostly calibration/losses.

In [ ]:
def fit_scalar_train_test(df, x_col, y_col, split_frac=0.7):
    n = len(df)
    n_train = int(n * split_frac)

    tr = df.iloc[:n_train].copy()
    te = df.iloc[n_train:].copy()

    x_tr = tr[x_col].to_numpy()
    y_tr = tr[y_col].to_numpy()
    x_te = te[x_col].to_numpy()
    y_te = te[y_col].to_numpy()

    # Least-squares scalar fit: y ~= k*x
    denom = np.dot(x_tr, x_tr)
    k = np.dot(x_tr, y_tr) / denom if denom > 0 else 1.0

    tr["pred_raw"] = x_tr
    tr["pred_scaled"] = k * x_tr
    te["pred_raw"] = x_te
    te["pred_scaled"] = k * x_te

    def metrics(y, p):
        err = p - y
        rmse = float(np.sqrt(np.mean(err**2)))
        mae = float(np.mean(np.abs(err)))
        bias = float(np.mean(err))
        r = float(pd.Series(p).corr(pd.Series(y)))
        r2 = r * r if np.isfinite(r) else np.nan
        return {"rmse": rmse, "mae": mae, "bias": bias, "r2": r2}

    return {
        "k": float(k),
        "train_raw": metrics(y_tr, tr["pred_raw"].to_numpy()),
        "train_scaled": metrics(y_tr, tr["pred_scaled"].to_numpy()),
        "test_raw": metrics(y_te, te["pred_raw"].to_numpy()),
        "test_scaled": metrics(y_te, te["pred_scaled"].to_numpy()),
        "train_df": tr,
        "test_df": te,
    }


# --- Test A power mapping ---
a_pow = pd.merge(
    wfa[["timestamp", "cf_model"]],
    pt[["timestamp", "generation_mwh"]],
    on="timestamp", how="inner"
).dropna().sort_values("timestamp").reset_index(drop=True)

a_pow["model_mwh_raw"] = a_pow["cf_model"] * PT_WFA_CAP_MW

a_sat = 100.0 * (a_pow["model_mwh_raw"] >= 0.99 * PT_WFA_CAP_MW).mean()
print("Test A power mapping")
print(f"  Raw saturation near rated: {a_sat:.1f}% of hours")
print(f"  Raw mean model={a_pow.model_mwh_raw.mean():.2f}, actual={a_pow.generation_mwh.mean():.2f}")

res_a = fit_scalar_train_test(a_pow, "model_mwh_raw", "generation_mwh")
print(f"  Fit scale k (train): {res_a['k']:.3f}")
print(f"  Test R² raw={res_a['test_raw']['r2']:.3f}  scaled={res_a['test_scaled']['r2']:.3f}")
print(f"  Test bias raw={res_a['test_raw']['bias']:.2f}  scaled={res_a['test_scaled']['bias']:.2f} MWh/h")


# --- Test B power mapping ---
b_pow = pd.merge(
    av[["timestamp", "cf_model"]],
    de[["timestamp", "generation_mwh", "cap_gw"]],
    on="timestamp", how="inner"
).dropna().sort_values("timestamp").reset_index(drop=True)

b_pow["model_mwh_raw"] = b_pow["cf_model"] * b_pow["cap_gw"] * 1000.0
b_sat = 100.0 * (b_pow["cf_model"] >= 0.99).mean()

print("\nTest B power mapping")
print(f"  Raw saturation near CF=1: {b_sat:.1f}% of hours")
print(f"  Raw mean model={b_pow.model_mwh_raw.mean():.2f}, actual={b_pow.generation_mwh.mean():.2f}")

res_b = fit_scalar_train_test(b_pow, "model_mwh_raw", "generation_mwh")
print(f"  Fit scale k (train): {res_b['k']:.3f}")
print(f"  Test R² raw={res_b['test_raw']['r2']:.3f}  scaled={res_b['test_scaled']['r2']:.3f}")
print(f"  Test bias raw={res_b['test_raw']['bias']:.2f}  scaled={res_b['test_scaled']['bias']:.2f} MWh/h")

In [ ]:
# Plot first 2 weeks: raw vs scaled vs actual (Test A and B)
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=False)

plot_a = a_pow.head(24 * 14).copy()
plot_a["scaled_mwh"] = res_a["k"] * plot_a["model_mwh_raw"]
axes[0].plot(plot_a["timestamp"], plot_a["generation_mwh"], label="Actual", color="black", lw=1.2)
axes[0].plot(plot_a["timestamp"], plot_a["model_mwh_raw"], label="Model raw", color="tomato", lw=1.0, alpha=0.8)
axes[0].plot(plot_a["timestamp"], plot_a["scaled_mwh"], label=f"Model scaled (k={res_a['k']:.2f})", color="steelblue", lw=1.0)
axes[0].set_title("Test A (WFA): first 2 weeks power mapping")
axes[0].set_ylabel("MWh/h")
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=8)

plot_b = b_pow.head(24 * 14).copy()
plot_b["scaled_mwh"] = res_b["k"] * plot_b["model_mwh_raw"]
axes[1].plot(plot_b["timestamp"], plot_b["generation_mwh"], label="Actual", color="black", lw=1.2)
axes[1].plot(plot_b["timestamp"], plot_b["model_mwh_raw"], label="Model raw", color="tomato", lw=1.0, alpha=0.8)
axes[1].plot(plot_b["timestamp"], plot_b["scaled_mwh"], label=f"Model scaled (k={res_b['k']:.2f})", color="steelblue", lw=1.0)
axes[1].set_title("Test B (AV -> DE fleet): first 2 weeks power mapping")
axes[1].set_ylabel("MWh/h")
axes[1].set_xlabel("Date")
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=8)

for ax in axes:
    for label in ax.get_xticklabels():
        label.set_rotation(25)

plt.tight_layout()
plt.show()

## How to judge this run

- Test A should be the stricter single-farm check.
- Test B will naturally be noisier because one point (Alpha Ventus) represents a spatially distributed fleet.
- Daily CF correlation is often more meaningful than hourly for weather-model validation.